#  2D Vector Addition
## Name: Innara
## Reg No: FA22-BCS-059


In [ ]:
%%writefile vecAdd.cu
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>
#include <math.h>

#define M 8
#define N 9

__global__
void vecAddKernel(int* A, int* B, int* C, int m, int n)
{
    int i = blockIdx.y * blockDim.y + threadIdx.y;
    int j = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < m && j < n)
    {
        C[i * n + j] = A[i * n + j] + B[i * n + j];
    }
}

__host__
void vecAdd(int* h_A, int* h_B, int* h_C, int m, int n)
{
    int *d_A, *d_B, *d_C;
    int size = m * n * sizeof(int);

    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    dim3 DimBlock(16, 16, 1);

    dim3 DimGrid(
        (int)ceil((float)n / 16),
        (int)ceil((float)m / 16),
        1
    );

    vecAddKernel<<<DimGrid, DimBlock>>>(d_A, d_B, d_C, m, n);

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

int main()
{
    int m = M;
    int n = N;

    int h_A[M][N];
    int h_B[M][N];
    int h_C[M][N];

    srand(time(0));

    for (int i = 0; i < m; i++)
    {
        for (int j = 0; j < n; j++)
        {
            h_A[i][j] = rand() % 10;
            h_B[i][j] = rand() % 10;
        }
    }

    vecAdd((int*)h_A, (int*)h_B, (int*)h_C, m, n);

    printf("Result:\n");
    for (int i = 0; i < m; i++)
    {
        for (int j = 0; j < n; j++)
        {
            printf("%d ", h_C[i][j]);
        }
        printf("\n");
    }

    return 0;
}

Overwriting vecAdd.cu


In [ ]:
!nvcc -arch=sm_75 vecAdd.cu -o vecAdd

In [ ]:
!./vecAdd

Result:
17 12 11 10 11 11 8 9 14 
12 4 14 5 2 5 4 9 6 
6 11 9 12 14 8 7 3 2 
16 13 13 10 8 9 3 18 1 
8 11 10 10 9 4 2 6 5 
10 14 9 9 7 18 2 8 15 
3 12 11 12 2 3 5 5 9 
10 13 15 6 11 8 6 9 1 
